In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.nn.conv import GATConv, GCNConv, SplineConv
from torch_geometric.nn.norm import GraphNorm
from torch_geometric.nn import Sequential
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
import torch.utils.data as torchdata
import tqdm
from math import floor
import gc
import matplotlib.pyplot as plt
from Model.MENDR.MENDR import MENDR_model
from Model.MENDR.MENDREncoder import MENDRPatchEncoder
from Model.MENDR.mAtt.optimizer import MixOptimizer
from Model.MENDR.mAtt.spd import SPDTangentSpace
import random
import os
import math
from Datasets.datasetTUEV import WaveletTUEVDataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve, auc
from torchjd import mtl_backward
from torchjd.aggregation import UPGrad

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
print(device)

cpu


In [3]:
print("*" * 50)
finetune_train_dataset = WaveletTUEVDataset(root="/home/hice1/mchen439/scratch/TUEV/train", frac=1.0, split="train")
print("Dataset Loaded. Length of Train Dataset: ", len(finetune_train_dataset))
finetune_eval_dataset = WaveletTUEVDataset(root="/home/hice1/mchen439/scratch/TUEV/eval", frac=1.0, split="eval")
print("Dataset Loaded. Length of Validation Dataset: ", len(finetune_eval_dataset))

**************************************************
Loading 83932 folders


100%|██████████| 83932/83932 [02:41<00:00, 520.61it/s]


Dataset Loaded. Length of Train Dataset:  83932
Loading 29421 folders


100%|██████████| 29421/29421 [00:58<00:00, 503.00it/s]

Dataset Loaded. Length of Validation Dataset:  29421


In [4]:
print(finetune_eval_dataset[0]['graph'])
print(finetune_eval_dataset[0]['delta'].shape)
print(finetune_eval_dataset[0]['theta'].shape)
print(finetune_eval_dataset[0]['alpha'].shape)
print(finetune_eval_dataset[0]['beta'].shape)
print(finetune_eval_dataset[0]['gamma'].shape)
print(finetune_eval_dataset[0]['graph'].y)
print(finetune_eval_dataset[1]['graph'].y)
print(finetune_eval_dataset[2]['graph'].y)
print(finetune_eval_dataset[3]['graph'].y)
print(finetune_eval_dataset[4]['graph'].y)
print(finetune_eval_dataset[5]['graph'].y)
print(finetune_eval_dataset[6]['graph'].y)
print(finetune_eval_dataset[7]['graph'].y)
print(finetune_eval_dataset[8]['graph'].y)
print(finetune_eval_dataset[9]['graph'].y)
print(finetune_eval_dataset[29000]['graph'].y)
print(finetune_eval_dataset[29010]['graph'].y)
print(finetune_eval_dataset[29020]['graph'].y)
print(finetune_eval_dataset[29030]['graph'].y)

Data(edge_index=[2, 361], edge_attr=[361, 1], pos=[19, 3], y=[1])
torch.Size([19, 46])
torch.Size([19, 46])
torch.Size([19, 86])
torch.Size([19, 166])
torch.Size([19, 325])
tensor([5])
tensor([5])
tensor([5])
tensor([5])
tensor([5])
tensor([2])
tensor([2])
tensor([0])
tensor([5])
tensor([5])
tensor([5])
tensor([0])
tensor([5])
tensor([1])


In [5]:
finetune_train_loader = DataLoader(finetune_train_dataset, batch_size=2, num_workers=1, shuffle=True, persistent_workers=True)
finetune_eval_loader = DataLoader(finetune_eval_dataset, batch_size=2, num_workers=1, shuffle=True, persistent_workers=True)
print(len(finetune_train_loader), len(finetune_eval_loader))
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']

41966 14711


In [8]:
class FinetuneDecoder(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.tangent = SPDTangentSpace(19, device=device)
        self.flatten = nn.Flatten()

        self.seq = nn.Sequential(nn.Linear(19 * 10, 19*10), nn.Dropout(p=0.1), nn.GELU(),
                                 nn.Linear(19*10, 19*10), nn.Dropout(p=0.1), nn.GELU())
        self.final_decoder = nn.Linear(19*10, 6)

    def forward(self, x):
        batch_size = x.shape[0]
        num_patches = x.shape[1]
        embedding_dim = x.shape[2]
        x = self.tangent(x.view(batch_size*num_patches, embedding_dim, embedding_dim))
        #x = x.view(batch_size, num_patches, -1)
        x = self.flatten(x)
        x = self.seq(x)
        return self.final_decoder(x)

In [9]:
model = MENDR_model(device=device).to(device)
#model.load_state_dict(torch.load("./checkpoint/f09fb583a9654a9582fa81ddc00b39c0_14_TINY/mendr_model_weights.pth", weights_only=True))
model_decoder = FinetuneDecoder(device).to(device)
optim_params = []
optim_params += list(model.parameters())
optim_params += list(model_decoder.parameters())
optimizer = MixOptimizer(torch.optim.AdamW(optim_params, lr=1e-3))
optimizer.set_scheduler_t0(len(finetune_train_loader))
for p in model.parameters():
    p.register_hook(lambda grad: torch.clamp(grad, -1e7, 1e7))
for p in model_decoder.parameters():
    p.register_hook(lambda grad: torch.clamp(grad, -1e7, 1e7))
aggregator = UPGrad(pref_vector=torch.tensor([10, 1, 1, 1, 1, 1]).to(device))
print("Total MENDR parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total model_decoder parameters:", sum(p.numel() for p in model_decoder.parameters() if p.requires_grad))
total_encoder_params = 0
total_decoder_params = 0
for band in BANDS:
    print(f'{band} Encoder Params: {model.mendr_encoder.encoder_decoders[band].getEncoderParamCount()}')
    print(f'{band} Decoder Params: {model.mendr_encoder.encoder_decoders[band].getDecoderParamCount()}')
    total_encoder_params += model.mendr_encoder.encoder_decoders[band].getEncoderParamCount()
    total_decoder_params += model.mendr_encoder.encoder_decoders[band].getDecoderParamCount()
#print(f'Wavelet Contextualizer Params: {sum(p.numel() for p in model.mendr_contextualizer.WaveletContextualizer.parameters() if p.requires_grad)}')
#print(f'Combined Contextualizer Params: {sum(p.numel() for p in model.mendr_contextualizer.CombinedContextualizer.parameters() if p.requires_grad)}')
print(f'Contextualizer Params: {sum(p.numel() for p in model.mendr_contextualizer.parameters() if p.requires_grad)}')
print("Total Encoder Params: ", total_encoder_params)
print("Total Decoder Params: ", total_decoder_params)

Set Cosine Annealing with Warm Restarts optimizer T_0 to: 41966
Total MENDR parameters: 2448254
Total model_decoder parameters: 73726
delta Encoder Params: 12597
delta Decoder Params: 22309
theta Encoder Params: 12597
theta Decoder Params: 22309
alpha Encoder Params: 14041
alpha Decoder Params: 101219
beta Encoder Params: 16929
beta Decoder Params: 498051
gamma Encoder Params: 22705
gamma Decoder Params: 1622675
Contextualizer Params: 70788
Total Encoder Params:  78869
Total Decoder Params:  2266563


In [10]:
def _fft_loss(output, target, loss_fn):
    hanning_window = torch.hann_window(output.shape[-1]).to(device)
    output_windowed = output * hanning_window.expand_as(output)
    target_windowed = target * hanning_window.expand_as(target)

    output_fft = torch.fft.fft(output_windowed, dim=-1)
    output_amplitude = torch.abs(output_fft)
    output_angle = torch.angle(output_fft)

    target_fft = torch.fft.fft(target_windowed, dim=-1)
    target_amplitude = torch.abs(target_fft)
    target_angle = torch.abs(target_fft)

    return (loss_fn(output_amplitude, target_amplitude) + (loss_fn(output_angle, target_angle)))

def calculate_band_loss(inputs, outputs, loss_fn, loss_type='real'):
    outputs_reshaped = outputs.reshape(inputs.shape[0], inputs.shape[1], inputs.shape[2], inputs.shape[3])
    if loss_type == 'real':
        loss = loss_fn(inputs, outputs_reshaped)
    elif loss_type == 'fourier':
        loss = _fft_loss(output_reshaped, inputs, loss_fn)
    else:
        loss = loss_fn(inputs, outputs_reshaped) + _fft_loss(output_reshaped, inputs, loss_fn) * 1e-2
    return loss

In [39]:
def train_one_epoch(model, epoch, decoder, optimizer, task_loss_fn, recon_loss_fn, train_loader):
	task_loss_sum = 0
	recon_loss_sum = 0
	model.train()
	decoder.train()

	pbar = tqdm.trange(len(train_loader), desc="Training")
	data_iterator = iter(train_loader)
	balanced_accuracies = []

	truths = []
	preds = []
	for iteration in pbar:
		data = next(data_iterator)
		relevant_bands = [data['delta'].float().to(device),
							data['theta'].float().to(device),
							data['alpha'].float().to(device),
							data['beta'].float().to(device),
							data['gamma'].float().to(device)]
		inputs = dict(zip(BANDS, relevant_bands))
		patchified_inputs, encodings, decodings, wavelet_manifold_output, combined_manifold_output = model(data['graph'], inputs)
		prediction = decoder(combined_manifold_output).float()
		ground_truth = data['graph'].y.to(device).long()

		task_loss = task_loss_fn(prediction, ground_truth)

		delta_loss = calculate_band_loss(patchified_inputs['delta'], decodings['delta'], recon_loss_fn)
		theta_loss = calculate_band_loss(patchified_inputs['theta'], decodings['theta'], recon_loss_fn)
		alpha_loss = calculate_band_loss(patchified_inputs['alpha'], decodings['alpha'], recon_loss_fn)
		beta_loss = calculate_band_loss(patchified_inputs['beta'],   decodings['beta'], recon_loss_fn)
		gamma_loss = calculate_band_loss(patchified_inputs['gamma'], decodings['gamma'], recon_loss_fn)

		# Borrowed from CBramod: https://github.com/wjq-learning/CBraMod/blob/e0002aaa87e77a954ce3a11cb1de1ea13e3585e2/finetune_evaluator.py
		with torch.no_grad():
			pred_y = torch.max(prediction, dim=-1)[1]
			truths += ground_truth.cpu().squeeze().numpy().tolist()
			preds += pred_y.cpu().squeeze().numpy().tolist()

		
		shared_features = [encoding for band, encoding in encodings.items()]
		optimizer.zero_grad()
		#task_loss.backward()
		mtl_backward(losses=[task_loss, delta_loss, theta_loss, alpha_loss, beta_loss, gamma_loss], features=shared_features, aggregator=aggregator)
		optimizer.step()
		optimizer.scheduler_step(epoch * len(train_loader) + iteration)
		task_loss_sum += task_loss.item()
		recon_loss = delta_loss.item() + theta_loss.item() + alpha_loss.item() + beta_loss.item() + gamma_loss.item()
		recon_loss_sum += recon_loss
		pbar.set_postfix(task_loss=task_loss.item(), recon_loss=recon_loss, accuracy=f'{accuracy*100:.3f}', balanced_accuracy=f'{balanced_accuracy*100:.3f}', AUROC=f'{roc_auc*100:.3f}', AUCPR=f'{pr_auc*100:.3f}', lr=optimizer.scheduler.get_last_lr()[0])

	truths = np.array(truths)
	preds = np.array(preds)

	balanced_accuracy = balanced_accuracy_score(truths, preds)
	accuracy = accuracy_score(truths, preds)
	roc_auc = roc_auc_score(truths, preds)
	precision, recall, thresholds = precision_recall_curve(truths, preds, pos_label=1)
	pr_auc = auc(recall, precision)
	print(f'Mean Acc: {balanced_accuracies} Acc: {accuracy} ROC AUC: {roc_auc} PR AUC: {pr_auc}')
	return task_loss_sum, recon_loss_sum

In [40]:
def validate_one_epoch(model, decoder, task_loss_fn, recon_loss_fn, val_loader):
	task_loss_sum = 0
	recon_loss_sum = 0
	model.eval()
	decoder.eval()

	pbar = tqdm.trange(len(val_loader), desc="Validating")
	data_iterator = iter(val_loader)
	balanced_accuracies = []
	auc_pr = []
	auroc = []

	with torch.no_grad():
		for iteration in pbar:
			data = next(data_iterator)
			relevant_bands = [data['delta'].float().to(device),
								data['theta'].float().to(device),
								data['alpha'].float().to(device),
								data['beta'].float().to(device),
								data['gamma'].float().to(device)]
			inputs = dict(zip(BANDS, relevant_bands))
			encoder_outputs = model(data['graph'], inputs)
			patchified_inputs, encodings, decodings, wavelet_manifold_output, combined_manifold_output = model(data['graph'], inputs)
			prediction = decoder(combined_manifold_output).float()
			ground_truth = data['graph'].y.to(device).float()
			ground_truth = torch.repeat_interleave(ground_truth, patchified_inputs['delta'].shape[1]) 

			task_loss = task_loss_fn(prediction, ground_truth[:, None])

			delta_loss = calculate_band_loss(patchified_inputs['delta'], decodings['delta'], recon_loss_fn)
			theta_loss = calculate_band_loss(patchified_inputs['theta'], decodings['theta'], recon_loss_fn)
			alpha_loss = calculate_band_loss(patchified_inputs['alpha'], decodings['alpha'], recon_loss_fn)
			beta_loss = calculate_band_loss(patchified_inputs['beta'],   decodings['beta'], recon_loss_fn)
			gamma_loss = calculate_band_loss(patchified_inputs['gamma'], decodings['gamma'], recon_loss_fn)

			score_y = torch.sigmoid(prediction)
			pred_y = torch.gt(score_y, 0.5).long().cpu().numpy()
			score_y = score_y.cpu().numpy()

			balanced_accuracy = balanced_accuracy_score(ground_truth.cpu().numpy(), pred_y)
			accuracy = accuracy_score(ground_truth.cpu().numpy(), pred_y)
			try:
				roc_auc = roc_auc_score(ground_truth.cpu().numpy(), score_y)
				precision, recall, thresholds = precision_recall_curve(ground_truth.cpu().numpy(), score_y, pos_label=1)
				pr_auc = auc(recall, precision)
			except:
				roc_auc = 0
				pr_auc = 0

			balanced_accuracies.append(balanced_accuracy)
			auc_pr.append([roc_auc, pr_auc])
			auroc.append(roc_auc)

			task_loss_sum += task_loss.item()
			recon_loss = delta_loss.item() + theta_loss.item() + alpha_loss.item() + beta_loss.item() + gamma_loss.item()
			recon_loss_sum += recon_loss

			pbar.set_postfix(task_loss=task_loss.item(), recon_loss=recon_loss, accuracy=f'{accuracy*100:.3f}', balanced_accuracy=f'{balanced_accuracy*100:.3f}', AUROC=f'{roc_auc*100:.3f}', AUCPR=f'{pr_auc*100:.3f}', lr=optimizer.scheduler.get_last_lr()[0])
		balanced_accuracies = np.array(balanced_accuracies)
		auc_pr = np.array(auc_pr)
		auroc = np.array(auroc)
		#balanced_accuracies.mean(), balanced_accuracies.std()
		#print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
	return task_loss_sum, recon_loss_sum, balanced_accuracies.mean(), balanced_accuracies.std(), auc_pr.mean(), auc_pr.std(), auroc.mean(), auroc.std()

In [42]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
torch.autograd.set_detect_anomaly(True)
for epoch_idx in range(1):
    training_task_loss, training_recon_loss = train_one_epoch(model, epoch_idx, model_decoder, optimizer, nn.CrossEntropyLoss(), nn.MSELoss(), finetune_train_loader)
    #eval_task_loss, eval_recon_loss, eval_acc, eval_std, eval_auc_pr, eval_auc_std, eval_auroc, eval_auroc_std = validate_one_epoch(model, model_decoder, nn.CrossEntropyLoss(), nn.MSELoss(), finetune_eval_loader)
    #print("*" * 50)
    #print(f"Epoch: {epoch_idx}")
    #print("Epoch: ", epoch_idx, "Total Training Task Loss: ", training_task_loss, "Total Training Recon Loss:", training_recon_loss, "Total Validation Task Loss: ", eval_task_loss, "Total Validation Recon Loss:", eval_recon_loss)
    #print("Epoch: ", epoch_idx, "Total Validation Task Loss: ", eval_task_loss, "Total Validation Recon Loss:", eval_recon_loss)
    #print("Epoch: ", epoch_idx, "Eval Acc", eval_acc, "Eval Std", eval_std, "Eval AUC PR", eval_auc_pr, "Eval AUC Std", eval_auc_std, "Eval AUROC", eval_auroc, "Eval AUROC Std", eval_auroc_std)

Training:   0%|          | 0/41966 [00:00<?, ?it/s]/home/hice1/mchen439/.local/lib/python3.10/site-packages/torch/autograd/graph.py:825: UserWarning: Error detected in MmBackward0. Traceback of forward call that caused the error:
  File "/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in sta

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [38, 38]], which is output 0 of AsStridedBackward0, is at version 1; expected version 0 instead. Hint: the backtrace further above shows the operation that failed to compute its gradient. The variable in question was changed in there or anywhere later. Good luck!